<a href="https://colab.research.google.com/github/api-sage/diaml_projects/blob/main/Project3/pafolabi_DIAML_Assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data, Inference, and Applied Machine Learning - Assignment 3

In [ ]:
#Importation of libraries
import numpy as np
import pandas as pd
import matplotlib as plt
import sklearn as sk

##Question 1

Dataset: [Michealson Speed of Light](https://raw.githubusercontent.com/api-sage/diaml_projects/refs/heads/main/Project3/Data/SpeedOfLight/michelson_speed_of_light.json)

In [ ]:
#Data ingestion
data_location = 'https://raw.githubusercontent.com/api-sage/diaml_projects/refs/heads/main/Project3/Data/SpeedOfLight/michelson_speed_of_light.json'
raw = pd.read_json(data_location)
#The measurements sit as a nested dictionary in the "observations" column, so I flatten that out into its own column
df = pd.json_normalize(raw['observations'])
#print(df.shape)// (20,2) Dataset contains 20 records and 2 features (run_id and measured_speed_kmpersec)

###(a) State appropriate null and alternative hypotheses for investigating whether Michelson's measurements deviate from the true value. Justify your choice of a one-tailed or two-tailed test.

**Defined hypothesis**:
1. NULL Hypothesis (Ho): µ = 299,734.5 km/s - Michelson's measurements do not systematically deviate from the true value of the speed of light in the air.

2. Alternative Hypothesis (H1): µ ≠ 299,734.5 km/s - Michelson's measurements systematically deviate from the true value of the speed of light in the air.

**Justification for a two-tailed test**

A two-tailed test is appropriate because I want to examine whether Michelson's measurements systematically deviate (in either direction) from the true value of the speed of light in the air. Before analysing the data, there is no physical or theoretical evidence to assume that his expereimental apparatus would err strictly on the left or right side. Therefore, a deviation of the measurement from the true value represents evidence against the accuracy of the measurement.




###(b) Given the sample size from the dataset, discuss how you would assess whether the normality assumption is reasonable, and what the consequences would be for your test if it isn't.

In [ ]:
#Checking normality with a Shapiro-Wilk test and a look at the shape of the data
from scipy import stats

speeds = df['measured_speed_kmpersec']
shapiro_stat, shapiro_p = stats.shapiro(speeds)
print("Shapiro-Wilk statistic:", shapiro_stat)
print("Shapiro-Wilk p-value:", shapiro_p)
print("Skewness:", speeds.skew())
print("Kurtosis:", speeds.kurt())

With only 20 measurements, I cannot rely on the central limit theorem to save me if the data are far from normal, so I need to actually check the shape of the data before trusting the t-test. The way I would go about this is to first look at a histogram and a QQ plot to see if the points sit roughly along a straight line and if the distribution looks bell shaped, and then back that up with a formal Shapiro-Wilk test since the sample is small enough for this test to be reliable.

From the numbers above, the Shapiro-Wilk p-value comes out well above 0.05, so I do not have enough evidence to say the data depart from normality. The skewness and kurtosis values are also not extreme, which supports the idea that a normal assumption is reasonable here even though the sample is small.

If it turned out that the data were clearly not normal, for example with a p-value below 0.05 or a strongly skewed histogram, the consequence would be that the sampling distribution of the mean may not be well approximated by a t-distribution when n is this small. This would make the p-value and confidence interval from a standard t-test unreliable. In that case I would use a non-parametric alternative such as the Wilcoxon signed-rank test, or a bootstrap approach to estimate the sampling distribution of the mean directly, instead of trusting the t-test results as they stand.

###(c) By using an appropriate statistical test, report the sample and population means, sample standard deviation, standard error of the mean (SEM), t statistic, degrees of freedom, and p-value. Explain whether the null hypothesis is rejected or not.

In [ ]:
#One sample t-test comparing the sample mean to the accepted speed of light in air
population_mean = 299734.5

sample_mean = speeds.mean()
sample_sd = speeds.std(ddof=1)
n = len(speeds)
sem = sample_sd / np.sqrt(n)
df_t = n - 1

t_stat, p_value = stats.ttest_1samp(speeds, population_mean)

print("Sample mean:", sample_mean)
print("Population mean (accepted value):", population_mean)
print("Sample standard deviation:", sample_sd)
print("Standard error of the mean:", sem)
print("t statistic:", t_stat)
print("Degrees of freedom:", df_t)
print("p-value:", p_value)

The sample mean works out to about 299,909 km/s compared to the accepted value of 299,734.5 km/s, with a sample standard deviation of about 104.9 km/s and a standard error of the mean of about 23.5 km/s. This gives a t statistic of about 7.44 on 19 degrees of freedom, and a p-value that is far smaller than 0.05 (in fact well below 0.001).

Since the p-value is much smaller than my chosen significance level of 0.05, I reject the null hypothesis. This means the data give strong evidence that Michelson's average measurement is not equal to the accepted value of the speed of light in air, and his experiment shows a real, systematic deviation from the true value.

###(d) Calculate the bias in km/s and as a percentage of the accepted value. Discuss whether statistical significance means the bias is scientifically important. Then recompute the SEM, t statistic, and p-value assuming n = 200 (same mean and SD), and discuss whether this changes your confidence in the result.

In [ ]:
#Bias in km/s and as a percentage of the accepted value
bias_kms = sample_mean - population_mean
bias_pct = (bias_kms / population_mean) * 100

print("Bias (km/s):", bias_kms)
print("Bias (% of accepted value):", bias_pct)

#Recomputing SEM, t statistic and p-value assuming n = 200, keeping the same mean and SD
n2 = 200
sem_n200 = sample_sd / np.sqrt(n2)
t_n200 = (sample_mean - population_mean) / sem_n200
df_n200 = n2 - 1
p_n200 = 2 * (1 - stats.t.cdf(abs(t_n200), df_n200))

print("SEM (n=200):", sem_n200)
print("t statistic (n=200):", t_n200)
print("Degrees of freedom (n=200):", df_n200)
print("p-value (n=200):", p_n200)

The bias here is about 174.5 km/s, which is about 0.058% of the accepted value. On its own that percentage looks tiny, so even though the test is statistically significant, I do not think this bias is scientifically important in a practical sense. Statistical significance in this case is mostly telling me that the measurements are precise and consistent enough for even a small, real offset to stand out, not that the offset itself is large or has real world consequences. In physics terms, a 0.06% error in a 19th century optical experiment is actually a very good result, so the significant p-value reflects precision rather than a meaningful practical error.

When I recompute the numbers assuming n = 200 while keeping the same mean and standard deviation, the SEM drops sharply to around 7.4 km/s, which pushes the t statistic up to about 23.5 and the p-value effectively down to zero. This does not change how confident I am that a real bias exists, since the test was already clearly significant with n = 20. What it does show is that as sample size grows, the test becomes more and more sensitive to tiny differences from the accepted value, which reinforces the point that a very small p-value does not by itself tell me anything about how big or important the bias is. This is exactly why I look at the bias size and percentage alongside the p-value rather than relying on statistical significance on its own.